In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))  # raíz de backend/
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [6]:
import logging
logging.basicConfig(level=logging.INFO)

from data_engine.services.ingestion.tasa_interes_mercado import ingerir
ingerir()

INFO:data_engine.services.ingestion.tasa_interes_mercado:Archivos: ['BCCh Tasas bonos pesos mensual.xlsx', 'BCCh Tasas bonos UF mensual.xlsx', 'BCCh TPM mensual.xlsx', 'BCCh Tasas PRC PRBC mensual.xlsx', 'BCCh Tasas captacion pesos mensual.xlsx', 'BCCh Tasas captacion UF mensual.xlsx']
INFO:data_engine.services.ingestion.tasa_interes_mercado:Tasas de interés (BCCh): {'creados': 0, 'actualizados': 0, 'sin_cambios': 6786}


{'creados': 0, 'actualizados': 0, 'sin_cambios': 6786}

In [7]:
from django.db.models import Count, Min, Max
from data_engine.models import TasaInteresMercado

list(
    TasaInteresMercado.objects
    .values("instrumento", "plazo_anios", "tipo")
    .annotate(n=Count("id"), desde=Min("fecha"), hasta=Max("fecha"))
    .order_by("instrumento", "plazo_anios", "tipo")
)

[{'instrumento': 'bono_pesos',
  'plazo_anios': 1,
  'tipo': 'nominal',
  'n': 240,
  'desde': datetime.date(2003, 3, 1),
  'hasta': datetime.date(2025, 11, 1)},
 {'instrumento': 'bono_pesos',
  'plazo_anios': 2,
  'tipo': 'nominal',
  'n': 270,
  'desde': datetime.date(2002, 9, 1),
  'hasta': datetime.date(2026, 9, 1)},
 {'instrumento': 'bono_pesos',
  'plazo_anios': 5,
  'tipo': 'nominal',
  'n': 280,
  'desde': datetime.date(2002, 9, 1),
  'hasta': datetime.date(2026, 9, 1)},
 {'instrumento': 'bono_pesos',
  'plazo_anios': 10,
  'tipo': 'nominal',
  'n': 267,
  'desde': datetime.date(2004, 7, 1),
  'hasta': datetime.date(2026, 9, 1)},
 {'instrumento': 'bono_uf',
  'plazo_anios': 1,
  'tipo': 'real',
  'n': 205,
  'desde': datetime.date(2006, 2, 1),
  'hasta': datetime.date(2025, 11, 1)},
 {'instrumento': 'bono_uf',
  'plazo_anios': 2,
  'tipo': 'real',
  'n': 227,
  'desde': datetime.date(2005, 1, 1),
  'hasta': datetime.date(2026, 9, 1)},
 {'instrumento': 'bono_uf',
  'plazo_anios'

In [8]:
# Reingesta de tasas con las series históricas nuevas (PRC, PRBC y captación).
# Esperado: 3.864 creados (las nuevas), 0 actualizados y 2.922 sin cambios (las que ya estaban).
import logging
logging.basicConfig(level=logging.INFO)

from data_engine.services.ingestion.tasa_interes_mercado import ingerir
ingerir()

INFO:data_engine.services.ingestion.tasa_interes_mercado:Archivos: ['BCCh Tasas bonos pesos mensual.xlsx', 'BCCh Tasas bonos UF mensual.xlsx', 'BCCh TPM mensual.xlsx', 'BCCh Tasas PRC PRBC mensual.xlsx', 'BCCh Tasas captacion pesos mensual.xlsx', 'BCCh Tasas captacion UF mensual.xlsx']
INFO:data_engine.services.ingestion.tasa_interes_mercado:Tasas de interés (BCCh): {'creados': 0, 'actualizados': 0, 'sin_cambios': 6786}


{'creados': 0, 'actualizados': 0, 'sin_cambios': 6786}

In [10]:
# Resumen de lo guardado: cantidad de meses y rango de fechas por serie.
import pandas as pd
from django.db.models import Count, Min, Max
from data_engine.models import TasaInteresMercado

pd.DataFrame(
    TasaInteresMercado.objects
    .values("nombre", "tipo")
    .annotate(meses=Count("id"), desde=Min("fecha"), hasta=Max("fecha"))
    .order_by("nombre", "tipo")
)

,nombre,tipo,meses,desde,hasta
0,Bono en pesos a 1 año,nominal,240,2003-03-01,2025-11-01
1,Bono en pesos a 10 años,nominal,267,2004-07-01,2026-09-01
2,Bono en pesos a 2 años,nominal,270,2002-09-01,2026-09-01
3,Bono en pesos a 5 años,nominal,280,2002-09-01,2026-09-01
4,Bono en UF a 1 año,real,205,2006-02-01,2025-11-01
5,Bono en UF a 10 años,real,289,2002-09-01,2026-09-01
6,Bono en UF a 2 años,real,227,2005-01-01,2026-09-01
7,Bono en UF a 20 años,real,287,2002-09-01,2026-07-01
8,Bono en UF a 30 años,real,222,2008-03-01,2026-09-01
9,Bono en UF a 5 años,real,280,2002-09-01,2026-09-01
